# Model Evaluation and Comparison

This notebook compares Model A (SVM) and Model B (KNN) using:
- Side-by-side bar chart of accuracy, precision, recall, and F1 score
- Confusion matrix analysis
- ROC curve and AUC scores
- Dataset property analysis
- Final verdict on the better model

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    confusion_matrix, roc_auc_score, RocCurveDisplay
)

# Set seed for reproducibility
RANDOM_STATE = 42
sns.set_style('whitegrid')

In [ ]:
# Record library versions for reproducibility
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import seaborn as sns

print("="*60)
print("LIBRARY VERSIONS (for reproducibility)")
print("="*60)
print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")
try:
    print(f"Random State: {RANDOM_STATE}")
except NameError:
    print("Random State: Not defined yet")
print("="*60)

## 1. Load and Prepare Data

In [ ]:
# Load dataset
df = pd.read_csv('my_data .csv')

# Split features and target
X = df.drop(columns=['placed'])
y = df['placed']

# Identify feature types
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Dataset shape: {df.shape}")
print(f"Number of numerical features: {len(num_features)}")
print(f"Number of categorical features: {len(cat_features)}")
print(f"Class distribution:\n{y.value_counts()}")

# Stratified train/test split (25% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f"\nTraining set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

## 2. Define Preprocessing Pipeline

In [ ]:
# Define transformers
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ]
)

## 3. Define and Train Models

In [ ]:
# Model A: SVM with RBF kernel
model_a = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_STATE, probability=True))
])

# Model B: KNN (KNeighborsClassifier) with GridSearch
knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(weights="distance"))
])

param_grid_knn = {
    'classifier__n_neighbors': [3, 5, 7, 9]
}
model_b_cv = GridSearchCV(knn_pipeline, param_grid_knn, cv=5)

# Train models
print("Training Model A (SVM)...")
model_a.fit(X_train, y_train)
print("Model A trained successfully!")

print("\nTraining Model B (KNN with GridSearch)...")
model_b_cv.fit(X_train, y_train)
print(f"Model B trained successfully!")
print(f"Best parameters: {model_b_cv.best_params_}")

## 4. Evaluate Models and Collect Metrics

In [ ]:
def get_metrics(model, X_test, y_test, model_name):
    """Calculate all evaluation metrics for a model"""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_proba) if y_proba is not None else 0,
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp
    }
    
    print(f"\n{'='*50}")
    print(f"{model_name} Metrics")
    print(f"{'='*50}")
    for k, v in metrics.items():
        if k in ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC']:
            print(f"{k:15s}: {v:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"  TN: {tn:4d}  |  FP: {fp:4d}")
    print(f"  FN: {fn:4d}  |  TP: {tp:4d}")
    
    return metrics

# Evaluate both models
results_a = get_metrics(model_a, X_test, y_test, "Model A (SVM)")
results_b = get_metrics(model_b_cv, X_test, y_test, "Model B (KNN)")

## 5. Side-by-Side Comparison Plot

In [ ]:
# Create comparison plot
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
df_plot = pd.DataFrame({
    'Metric': metrics_to_plot * 2,
    'Value': [results_a[m] for m in metrics_to_plot] + [results_b[m] for m in metrics_to_plot],
    'Model': ['Model A (SVM)'] * 4 + ['Model B (KNN)'] * 4
})

plt.figure(figsize=(12, 7))
ax = sns.barplot(x='Metric', y='Value', hue='Model', data=df_plot, palette=['#2E86AB', '#A23B72'])
plt.title('Model Performance Comparison: SVM vs KNN', fontsize=16, fontweight='bold', pad=20)
plt.ylim(0, 1.05)
plt.ylabel('Score', fontsize=12)
plt.xlabel('Metric', fontsize=12)
plt.legend(title='Model', fontsize=11, title_fontsize=12)

# Add value labels on bars
for p in ax.patches:
    ax.annotate(f'{p.get_height():.3f}', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', fontsize=10, color='black', 
                xytext=(0, 8), textcoords='offset points')

plt.tight_layout()
plt.savefig('comparison_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Comparison plot saved as 'comparison_plot.png'")

## 6. ROC Curves Comparison

In [ ]:
# Plot ROC curves for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model A ROC
RocCurveDisplay.from_estimator(model_a, X_test, y_test, ax=axes[0], color='#2E86AB')
axes[0].plot([0, 1], [0, 1], linestyle='--', lw=2, color='gray', alpha=0.6)
axes[0].set_title(f'ROC Curve - Model A (SVM)\nAUC = {results_a["AUC"]:.4f}', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)

# Model B ROC
RocCurveDisplay.from_estimator(model_b_cv, X_test, y_test, ax=axes[1], color='#A23B72')
axes[1].plot([0, 1], [0, 1], linestyle='--', lw=2, color='gray', alpha=0.6)
axes[1].set_title(f'ROC Curve - Model B (KNN)\nAUC = {results_b["AUC"]:.4f}', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_curves_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ ROC curves saved as 'roc_curves_comparison.png'")

## 7. Confusion Matrix Comparison

In [ ]:
# Plot confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Model A Confusion Matrix
cm_a = confusion_matrix(y_test, model_a.predict(X_test))
sns.heatmap(cm_a, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title('Model A (SVM)\nConfusion Matrix', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Actual', fontsize=11)
axes[0].set_xlabel('Predicted', fontsize=11)

# Model B Confusion Matrix
cm_b = confusion_matrix(y_test, model_b_cv.predict(X_test))
sns.heatmap(cm_b, annot=True, fmt='d', cmap='Purples', ax=axes[1], cbar=False)
axes[1].set_title('Model B (KNN)\nConfusion Matrix', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Actual', fontsize=11)
axes[1].set_xlabel('Predicted', fontsize=11)

plt.tight_layout()
plt.savefig('confusion_matrices_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Confusion matrices saved as 'confusion_matrices_comparison.png'")

## 8. Model Justification and Verdict

### Analysis Framework:
1. **Metric Balance** (Precision vs Recall Trade-off)
2. **Confusion Matrix Errors** (False Negatives/Positives)
3. **ROC Curve Shape/AUC**
4. **Dataset Properties** (Feature scales, nonlinearity, dimensionality)

In [ ]:
# Generate detailed justification
print("="*80)
print("MODEL COMPARISON JUSTIFICATION")
print("="*80)

# Metric comparison
print("\n1. METRIC BALANCE (Precision vs Recall):")
print(f"   Model A (SVM): Precision={results_a['Precision']:.4f}, Recall={results_a['Recall']:.4f}")
print(f"   Model B (KNN): Precision={results_b['Precision']:.4f}, Recall={results_b['Recall']:.4f}")
print(f"   → Precision-Recall difference: SVM={abs(results_a['Precision']-results_a['Recall']):.4f}, KNN={abs(results_b['Precision']-results_b['Recall']):.4f}")

# Confusion matrix analysis
print("\n2. CONFUSION MATRIX ERRORS:")
print(f"   Model A (SVM): FP={results_a['FP']}, FN={results_a['FN']} (Total errors: {results_a['FP']+results_a['FN']})")
print(f"   Model B (KNN): FP={results_b['FP']}, FN={results_b['FN']} (Total errors: {results_b['FP']+results_b['FN']})")

# ROC/AUC comparison
print("\n3. ROC CURVE & AUC:")
print(f"   Model A (SVM): AUC={results_a['AUC']:.4f}")
print(f"   Model B (KNN): AUC={results_b['AUC']:.4f}")
print(f"   → AUC difference: {abs(results_a['AUC']-results_b['AUC']):.4f}")

# Dataset properties
print("\n4. DATASET PROPERTIES:")
print(f"   - Features: {len(num_features)} numerical, {len(cat_features)} categorical")
print(f"   - Total dimensionality after encoding: ~{X_train.shape[1]} → higher after one-hot encoding")
print(f"   - Feature scales: Mixed (requires StandardScaler)")
print(f"   - SVM with RBF kernel: Handles nonlinearity via kernel trick, robust to high dimensions")
print(f"   - KNN: Distance-based, sensitive to feature scaling and curse of dimensionality")

# Determine better model
better_model = "Model A (SVM)" if results_a['F1 Score'] > results_b['F1 Score'] else "Model B (KNN)"
better_results = results_a if results_a['F1 Score'] > results_b['F1 Score'] else results_b

print("\n" + "="*80)
print("FINAL VERDICT")
print("="*80)
print(f"\n🏆 WINNER: {better_model}")
print(f"\nJustification:")

if better_model == "Model A (SVM)":
    justification = f"""
Model A (SVM with RBF kernel) is the better choice for this dataset with an F1 score of {results_a['F1 Score']:.4f} 
compared to KNN's {results_b['F1 Score']:.4f}. The SVM achieves superior precision-recall balance 
(Precision: {results_a['Precision']:.4f}, Recall: {results_a['Recall']:.4f}), indicating robust performance across 
both positive and negative classes. The confusion matrix reveals fewer total errors ({results_a['FP']+results_a['FN']} 
vs {results_b['FP']+results_b['FN']}), with the RBF kernel effectively capturing nonlinear decision boundaries. 
The ROC curve demonstrates stronger discriminative power (AUC: {results_a['AUC']:.4f} vs {results_b['AUC']:.4f}). 
Given the dataset's mixed feature scales and moderate-to-high dimensionality after one-hot encoding, SVM's kernel-based 
approach handles the feature space more effectively than KNN's distance-based method, which suffers from the curse of 
dimensionality. The SVM's ability to find optimal hyperplanes in transformed feature spaces makes it the superior model 
for this classification task.
    """
else:
    justification = f"""
Model B (KNN) is the better choice for this dataset with an F1 score of {results_b['F1 Score']:.4f} 
compared to SVM's {results_a['F1 Score']:.4f}. The KNN achieves superior precision-recall balance 
(Precision: {results_b['Precision']:.4f}, Recall: {results_b['Recall']:.4f}), indicating robust performance across 
both positive and negative classes. The confusion matrix reveals fewer total errors ({results_b['FP']+results_b['FN']} 
vs {results_a['FP']+results_a['FN']}), with the distance-weighted voting effectively capturing local patterns. 
The ROC curve demonstrates stronger discriminative power (AUC: {results_b['AUC']:.4f} vs {results_a['AUC']:.4f}). 
Despite the dataset's moderate dimensionality, the optimized k-value (k={model_b_cv.best_params_['classifier__n_neighbors']}) 
and distance weighting allow KNN to leverage local neighborhood structures effectively. The preprocessing pipeline's 
StandardScaler mitigates KNN's sensitivity to feature scales, enabling it to outperform the SVM's global decision boundary 
approach for this particular dataset's distribution.
    """

print(justification.strip())
print("\n" + "="*80)

## Summary

This notebook has:
1. ✓ Created a side-by-side bar chart comparing accuracy, precision, recall, and F1 score
2. ✓ Analyzed metric balance (precision vs recall trade-off)
3. ✓ Examined confusion matrix errors (false negatives/positives)
4. ✓ Evaluated ROC curve shape and AUC scores
5. ✓ Considered dataset properties (feature scales, nonlinearity, dimensionality)
6. ✓ Provided a clear verdict naming the better model with comprehensive justification